# ICU 24-Hour Vitals Trajectory Forecasting (`PyTorch` Bi-LSTM + MultiheadAttention)
Downloads real ICU `chartevents.csv.gz`, `icustays.csv.gz`, and `admissions.csv.gz` from PhysioNet MIMIC-IV Demo v2.2, builds `(N_stays, 24_hours, 5_vital_channels)` tensors `[HR, SpO2, SBP, RespRate, TempC]`, trains `VitalSignsBiLSTMAttentionNet`, and saves `vitals_bilstm_attention.pth`.

In [ ]:
# !pip install -q torch scikit-learn pandas numpy


[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [2]:
from __future__ import annotations
import urllib.request
from pathlib import Path
from typing import Any, Dict, Optional, Tuple
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score

## Download & Inspect PhysioNet MIMIC-IV Demo ICU Tables

In [3]:
MIMIC_BASE_URL = "https://physionet.org/files/mimic-iv-demo/2.2/"
data_dir = Path("./clinical_data_cache")
icu_dir = data_dir / "mimic_iv_demo"
icu_dir.mkdir(parents=True, exist_ok=True)

files = {
    "chartevents.csv.gz": "icu/chartevents.csv.gz",
    "icustays.csv.gz": "icu/icustays.csv.gz",
    "admissions.csv.gz": "hosp/admissions.csv.gz",
}
for fname, rel_path in files.items():
    fpath = icu_dir / fname
    if not fpath.exists():
        url = MIMIC_BASE_URL + rel_path
        print(f"Downloading PhysioNet MIMIC-IV Demo table: {url}")
        urllib.request.urlretrieve(url, fpath)

In [8]:
pd.read_csv("https://physionet.org/files/mimic-iv-demo/2.2/icu/chartevents.csv.gz").head()

,subject_id,hadm_id,stay_id,caregiver_id,charttime,storetime,itemid,value,valuenum,valueuom,warning
0,10005817,20626031,32604416,6770.0,2132-12-16 00:00:00,2132-12-15 23:45:00,225054,On,NaN,NaN,0.0
1,10005817,20626031,32604416,6770.0,2132-12-16 00:00:00,2132-12-15 23:43:00,223769,100,100.0,%,0.0
2,10005817,20626031,32604416,6770.0,2132-12-16 00:00:00,2132-12-15 23:47:00,223956,Atrial demand,NaN,NaN,0.0
3,10005817,20626031,32604416,6770.0,2132-12-16 00:00:00,2132-12-15 23:47:00,224866,Yes,NaN,NaN,0.0
4,10005817,20626031,32604416,6770.0,2132-12-16 00:00:00,2132-12-15 23:45:00,227341,No,0.0,NaN,0.0


In [4]:
def load_mimic_iv_demo_vitals(seq_len: int = 24) -> Tuple[np.ndarray, np.ndarray]:
    icustays = pd.read_csv(icu_dir / "icustays.csv.gz")
    admissions = pd.read_csv(icu_dir / "admissions.csv.gz")
    vital_itemids = {220045: 0, 220277: 1, 220179: 2, 220210: 3, 223761: 4}

    chartevents = pd.read_csv(
        icu_dir / "chartevents.csv.gz",
        usecols=["stay_id", "charttime", "itemid", "valuenum"],
    )
    chartevents = chartevents[chartevents["itemid"].isin(vital_itemids.keys())].dropna(subset=["valuenum"])
    chartevents["charttime"] = pd.to_datetime(chartevents["charttime"])

    temp_mask = chartevents["itemid"] == 223761
    chartevents.loc[temp_mask, "valuenum"] = (chartevents.loc[temp_mask, "valuenum"] - 32.0) * (5.0 / 9.0)

    stay_meta = icustays.merge(
        admissions[["hadm_id", "hospital_expire_flag"]], on="hadm_id", how="left"
    )
    stay_meta["high_acuity"] = ((stay_meta["hospital_expire_flag"] == 1) | (stay_meta["los"] >= 3.5)).astype(int)

    default_vitals = np.array([82.0, 97.0, 120.0, 17.0, 37.0], dtype=np.float32)
    sequences, labels = [], []

    for _, row in stay_meta.iterrows():
        sid = row["stay_id"]
        sub = chartevents[chartevents["stay_id"] == sid].sort_values("charttime")
        if len(sub) < 10:
            continue
        t0 = sub["charttime"].iloc[0]
        sub["hour_bin"] = ((sub["charttime"] - t0).dt.total_seconds() // 3600).astype(int)
        sub = sub[(sub["hour_bin"] >= 0) & (sub["hour_bin"] < seq_len)]

        seq = np.tile(default_vitals, (seq_len, 1))
        mask = np.zeros((seq_len, 5), dtype=bool)
        for _, ev in sub.iterrows():
            h = int(ev["hour_bin"])
            c = vital_itemids[int(ev["itemid"])]
            seq[h, c] = float(ev["valuenum"])
            mask[h, c] = True

        for c in range(5):
            last_val = default_vitals[c]
            for h in range(seq_len):
                if mask[h, c]:
                    last_val = seq[h, c]
                else:
                    seq[h, c] = last_val

        sequences.append(seq)
        labels.append(int(row["high_acuity"]))

    X_seq = np.stack(sequences, axis=0).astype(np.float32)
    y_seq = np.array(labels, dtype=np.int64)
    print(f"Built {len(X_seq)} real 24h ICU vital-sign tensors from MIMIC-IV Demo | Shape: {X_seq.shape}")
    return X_seq, y_seq

X_seq, y_seq = load_mimic_iv_demo_vitals(seq_len=24)

Built 138 real 24h ICU vital-sign tensors from MIMIC-IV Demo | Shape: (138, 24, 5)


In [6]:
X_seq[0]

array([[ 73.      ,  98.      , 120.      ,  17.      ,  36.38889 ],
       [ 73.      ,  97.      , 120.      ,  19.      ,  36.38889 ],
       [ 70.      , 100.      , 120.      ,  21.      ,  36.38889 ],
       [ 74.      ,  97.      , 120.      ,  16.      ,  36.38889 ],
       [ 73.      ,  97.      , 120.      ,  20.      ,  36.38889 ],
       [ 75.      ,  99.      , 120.      ,  18.      ,  36.38889 ],
       [ 93.      ,  96.      , 120.      ,  23.      ,  36.38889 ],
       [ 80.      ,  99.      , 120.      ,  24.      ,  36.38889 ],
       [ 84.      ,  99.      , 120.      ,  21.      ,  36.38889 ],
       [ 86.      ,  98.      , 136.      ,  20.      ,  36.38889 ],
       [ 76.      ,  92.      , 136.      ,  22.      ,  36.38889 ],
       [ 72.      ,  91.      , 113.      ,  13.      ,  36.38889 ],
       [ 76.      ,  97.      , 119.      ,  26.      ,  36.666668],
       [ 72.      ,  94.      , 117.      ,  23.      ,  36.666668],
       [ 65.      ,  96.      , 10

In [7]:
y_seq[0]

np.int64(1)

## Define & Train `VitalSignsBiLSTMAttentionNet`

In [9]:
class VitalSignsBiLSTMAttentionNet(nn.Module):
    """PyTorch 1D-CNN + Bidirectional LSTM + Multi-Head Temporal Self-Attention Network."""

    def __init__(self, in_channels: int = 5, hidden_dim: int = 32, num_heads: int = 4):
        super().__init__()
        self.conv1d = nn.Sequential(
            nn.Conv1d(in_channels, hidden_dim, kernel_size=3, padding=1),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
        )
        self.bilstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.2,
        )
        self.attn = nn.MultiheadAttention(embed_dim=hidden_dim * 2, num_heads=num_heads, batch_first=True)
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim * 2),
            nn.Linear(hidden_dim * 2, 32),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
        )

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        x_conv = self.conv1d(x.transpose(1, 2)).transpose(1, 2)
        lstm_out, _ = self.bilstm(x_conv)
        attn_out, attn_weights = self.attn(lstm_out, lstm_out, lstm_out)
        pooled = attn_out.mean(dim=1)
        logits = self.classifier(pooled).squeeze(-1)
        temporal_importance = attn_weights.mean(dim=1)
        return logits, temporal_importance


class PyTorchVitalsTrainer:
    """Trains and runs inference with VitalSignsBiLSTMAttentionNet on GPU/CPU."""

    def __init__(self, epochs: int = 25, lr: float = 2e-3):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.epochs = epochs
        self.lr = lr
        self.net = VitalSignsBiLSTMAttentionNet().to(self.device)
        self.mean_: Optional[np.ndarray] = None
        self.std_: Optional[np.ndarray] = None

    def fit(self, X_seq: np.ndarray, y_seq: np.ndarray) -> Dict[str, float]:
        self.mean_ = X_seq.mean(axis=(0, 1), keepdims=True)
        self.std_ = np.where(X_seq.std(axis=(0, 1), keepdims=True) < 1e-6, 1.0, X_seq.std(axis=(0, 1), keepdims=True))
        X_norm = (X_seq - self.mean_) / self.std_

        dataset = TensorDataset(torch.tensor(X_norm, dtype=torch.float32), torch.tensor(y_seq, dtype=torch.float32))
        loader = DataLoader(dataset, batch_size=16, shuffle=True)

        pos_weight = torch.tensor([(len(y_seq) - y_seq.sum()) / max(y_seq.sum(), 1.0)], device=self.device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = torch.optim.AdamW(self.net.parameters(), lr=self.lr, weight_decay=1e-4)

        self.net.train()
        for _ in range(self.epochs):
            for bx, by in loader:
                bx, by = bx.to(self.device), by.to(self.device)
                optimizer.zero_grad()
                logits, _ = self.net(bx)
                loss = criterion(logits, by)
                loss.backward()
                optimizer.step()

        self.net.eval()
        with torch.no_grad():
            all_x = torch.tensor(X_norm, dtype=torch.float32, device=self.device)
            logits, _ = self.net(all_x)
            probs = torch.sigmoid(logits).cpu().numpy()

        auc = float(roc_auc_score(y_seq, probs)) if len(np.unique(y_seq)) > 1 else 0.5
        print(f"PyTorch Bi-LSTM + MultiheadAttention trained on {self.device} | ICU Sequence ROC-AUC: {auc:.4f}")
        return {"sequence_roc_auc": round(auc, 4)}

    def forecast_trajectory(self, vitals_24h: np.ndarray) -> Dict[str, Any]:
        self.net.eval()
        x_norm = (vitals_24h[None, :, :] - self.mean_) / self.std_
        with torch.no_grad():
            tx = torch.tensor(x_norm, dtype=torch.float32, device=self.device)
            logit, attn_map = self.net(tx)
            prob = float(torch.sigmoid(logit).item())
            hour_weights = attn_map[0].cpu().numpy()

        peak_hours = [int(h) for h in np.argsort(hour_weights)[-3:][::-1]]
        slope = (vitals_24h[-1] - vitals_24h[-6]) / 5.0
        proj_6h = vitals_24h[-1] + 6.0 * 0.5 * slope
        return {
            "deterioration_6h_prob": round(prob, 4),
            "peak_attention_hours": peak_hours,
            "projected_vitals_6h": {
                "heart_rate_bpm": round(float(proj_6h[0]), 1),
                "spo2_pct": round(float(np.clip(proj_6h[1], 70.0, 100.0)), 1),
                "systolic_bp_mmhg": round(float(proj_6h[2]), 1),
                "resp_rate_bpm": round(float(proj_6h[3]), 1),
                "temp_c": round(float(proj_6h[4]), 1),
            },
        }

## Train & Save Checkpoint (`vitals_bilstm_attention.pth`)

In [10]:
vitals_trainer = PyTorchVitalsTrainer(epochs=25, lr=2e-3)
vitals_trainer.fit(X_seq, y_seq)

checkpoint = {
    "model_state_dict": vitals_trainer.net.state_dict(),
    "mean": vitals_trainer.mean_,
    "std": vitals_trainer.std_,
    "epochs": vitals_trainer.epochs,
    "lr": vitals_trainer.lr,
    "in_channels": 5,
    "hidden_dim": 32,
    "num_heads": 4,
}
torch.save(checkpoint, "vitals_bilstm_attention.pth")
print("Model saved successfully to vitals_bilstm_attention.pth!")
print("Sample trajectory forecast:", vitals_trainer.forecast_trajectory(X_seq[0]))

PyTorch Bi-LSTM + MultiheadAttention trained on cpu | ICU Sequence ROC-AUC: 0.9250
Model saved successfully to vitals_bilstm_attention.pth!
Sample trajectory forecast: {'deterioration_6h_prob': 0.464, 'peak_attention_hours': [10, 21, 20], 'projected_vitals_6h': {'heart_rate_bpm': 65.8, 'spo2_pct': 93.4, 'systolic_bp_mmhg': 106.2, 'resp_rate_bpm': 21.2, 'temp_c': 36.9}}
